<a href="https://colab.research.google.com/github/moalhadiri/BraudeauRichardsFEM-article/blob/main/run_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Braudeau–Richards examples

Choose your meshes, final times and time-step levels in the form, then select **Runtime → Run all**. The calculations and figures appear here. No project folder or source-file editing is needed.

Use a CPU runtime. Colab may ask you to sign in with Google. Additional settings for spatial validation and the physical examples are grouped below the main form.


## 1. Choose the parameters

Enter the subdivision counts, final time and comma-separated time-step indices. For example, $\ell=1,2,3,4$ gives $\tau=0.1,0.05,0.025,0.0125$. Final times for fixed-step examples must be multiples of their time steps.


In [1]:
# @title Validation settings
# @markdown Mesh values count subdivisions: n = 16 means h = 1/16.
temporal_mesh_2d = 9 # @param {type:"integer"}
temporal_mesh_3d = 9 # @param {type:"integer"}
validation_final_time = 1 # @param {type:"number"}
temporal_time_levels = "1, 2, 3" # @param {type:"string"}
# @markdown tau = 0.1 * 2**(1 - ell). Choose Runtime > Run all after editing.
calculation_succeeded = False


### Additional settings (optional)

Keep these small defaults, or change the spatial meshes and physical examples here.


In [2]:
# @title Additional settings (optional)
# @markdown **Spatial validation.** Comma-separated subdivision counts; the polynomial reference is stationary.
spatial_meshes_2d = "4, 8, 16" # @param {type:"string"}
spatial_meshes_3d = "4, 8" # @param {type:"string"}
spatial_time_level = 3 # @param {type:"integer"}
# @markdown **Slotted sphere.** Times are in hours; n subdivisions give h/L = 1/n.
sphere_mesh = 16 # @param {type:"integer"}
sphere_final_time = 0.1 # @param {type:"number"}
sphere_time_level = 2 # @param {type:"integer"}
# @markdown **Nonlinear Fourier example.** Its constitutive exponent ell_p stays equal to 3.
fourier_mesh = 8 # @param {type:"integer"}
fourier_final_time = 0.2 # @param {type:"number"}
fourier_time_level = 2 # @param {type:"integer"}
# @markdown **Dry infiltration.** Times are in hours; this case keeps adaptive time steps.
dry_mesh_x = 20 # @param {type:"integer"}
dry_mesh_z = 10 # @param {type:"integer"}
dry_final_time = 1.0 # @param {type:"number"}


## 2. Prepare the example

This step prepares the runtime automatically. No Google Drive access is needed.


In [3]:
from io import BytesIO
from pathlib import Path
import os
import stat
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

# Keep the checked numerical source for repeatable runs.
revision = "335f48b16c91c2807919c0a17b1107f0f9515e73"
cache = Path(tempfile.gettempdir()) / ("braudeau_richards_" + revision[:8])
project = cache / ("BraudeauRichardsFEM-" + revision)
required = (project / "test.py", project / "requirements.txt", project / "parameters.json", project / "source", project / "data")
if not all(path.exists() for path in required):
    cache.mkdir(parents=True, exist_ok=True)
    url = "https://codeload.github.com/moalhadiri/BraudeauRichardsFEM/zip/" + revision
    request = urllib.request.Request(url, headers={"User-Agent": "BraudeauRichardsFEM-example"})
    with urllib.request.urlopen(request, timeout=60) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(BytesIO(archive_bytes)) as archive:
        for entry in archive.infolist():
            destination = (cache / entry.filename).resolve()
            if "\\" in entry.filename or ".." in Path(entry.filename).parts or not destination.is_relative_to(cache.resolve()) or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Unexpected path in the source archive")
        archive.extractall(cache)
if not all(path.exists() for path in required):
    raise RuntimeError("The source archive is incomplete")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", str(project / "requirements.txt")], check=True)
print("Ready: source revision", revision[:8])


HTTPError: HTTP Error 404: Not Found

In [ ]:
# @title Check the selected settings
import json

calculation_succeeded = False

def integer_list(text):
    try:
        return [int(value.strip()) for value in text.split(",") if value.strip()]
    except (AttributeError, ValueError) as error:
        raise ValueError("Enter a comma-separated list of integers") from error

parameters = json.loads((project / "parameters.json").read_text(encoding="utf8"))
parameters["spatial"].update(meshes_2d=integer_list(spatial_meshes_2d), meshes_3d=integer_list(spatial_meshes_3d), ell=spatial_time_level, final_time=validation_final_time)
parameters["temporal"].update(mesh_2d=temporal_mesh_2d, mesh_3d=temporal_mesh_3d, ell=integer_list(temporal_time_levels), final_time=validation_final_time)
parameters["sphere"].update(mesh=sphere_mesh, ell=sphere_time_level, final_time=sphere_final_time)
parameters["fourier"].update(mesh=fourier_mesh, ell=fourier_time_level, final_time=fourier_final_time)
parameters["dry"].update(mesh_x=dry_mesh_x, mesh_z=dry_mesh_z, final_time=dry_final_time)
parameters_path = project / "reviewer_parameters.json"
parameters_path.write_text(json.dumps(parameters, indent=2) + "\n", encoding="utf8", newline="\n")
check = subprocess.run([sys.executable, str(project / "test.py"), "--parameters", str(parameters_path), "--check-parameters"], cwd=project, capture_output=True, text=True, encoding="utf8")
if check.returncode:
    raise ValueError(check.stderr.strip().splitlines()[-1])
checked = json.loads(check.stdout)
print("Selected temporal time steps:")
for ell, tau in checked["time_steps"]["temporal"].items():
    print(f"  ell = {ell}: tau = {tau:g}")
print("The meshes and final times are valid. Ready to calculate.")


## 3. Run the calculations

The form determines the meshes and final times. Each time-step index gives $\tau=0.1\times2^{1-\ell}$. Here $\ell$ is a time-step index; the Fourier constitutive exponent $\ell_p=3$ is unchanged.

The full numerical output appears below. Each run uses a fresh results folder. Change `workers` below only if you want another number of assembly workers.


In [ ]:
workers = 1
calculation_succeeded = False
results = project / "results"
results.mkdir(exist_ok=True)
output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=results))
command = [sys.executable, "-u", str(project / "test.py"), "--no-show", "--update-readme", "--parameters", str(parameters_path), "--workers", str(workers), "--output", str(output_dir)]
run_environment = dict(os.environ, BRAUDEAU_SOURCE_REVISION=revision)
process = subprocess.Popen(command, cwd=project, env=run_environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf8", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise
if return_code != 0:
    raise RuntimeError("A calculation or check failed; see the output above")
calculation_succeeded = True


## Results from this run

This table comes from the calculations above. This session also updates its own copy of the README. Updating the public GitHub page is reserved for the repository owner through **Actions → Actualiser les résultats → Run workflow**.


In [ ]:
from IPython.display import Markdown, display
import shutil

if not calculation_succeeded:
    raise RuntimeError("Run the calculations successfully before showing their table")
readme = (project / "README.md").read_text(encoding="utf8")
section = readme.split("<!-- BEGINEXAMPLERESULTS -->", 1)[1].split("<!-- ENDEXAMPLERESULTS -->", 1)[0]
table = "\n".join(line for line in section.splitlines() if line.startswith("|"))
display(Markdown(table))
for filename in ("latest_results.json", "latest_results.csv"):
    shutil.copyfile(project / "data" / filename, output_dir / filename)
snapshot = json.loads((project / "data/latest_results.json").read_text(encoding="utf8"))
print(f"All {len(snapshot['cases'])} cases are also saved in latest_results.json and latest_results.csv.")


## 4. View the figures

These images come from the fields just computed. They are also saved as PNG files beside `results.txt`.


In [ ]:
from IPython.display import Image, Markdown, display

if not calculation_succeeded:
    raise RuntimeError("Run the calculations successfully before showing their figures")
figures = [
    ("Spatial errors in 2D", "spatial_errors_2d.png"),
    ("Spatial errors in 3D", "spatial_errors_3d.png"),
    ("Temporal errors", "temporal_errors.png"),
    ("Computed 2D solution", "solution_2d.png"),
    ("Slotted wet sphere in 3D", "slotted_sphere.png"),
    ("Dry infiltration in 2D", "dry_infiltration.png"),
    ("Newton corrections", "newton_corrections.png"),
]
for title, filename in figures:
    path = output_dir / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    display(Markdown("### " + title))
    display(Image(filename=str(path), width=850))
print("Text summary:", output_dir / "results.txt")


## Optional: save the results

Set `download_results = True` and run this cell to download the text, seven images and JSON/CSV data together. Leaving it `False` keeps everything in the browser session.


In [ ]:
download_results = False

if download_results:
    import shutil
    if not calculation_succeeded:
        raise RuntimeError("No successful calculation is available to download")
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        export_folder = Path.cwd() / "results"
        export_folder.mkdir(exist_ok=True)
        export = export_folder / "braudeau_results.zip"
        shutil.copyfile(archive, export)
        display(FileLink("results/" + export.name))
    else:
        files.download(archive)
